In [ ]:
# !pip install soundfile
# !pip install torchsde

## STABILITY AI

In [ ]:
import torch
import soundfile as sf
from diffusers import StableAudioPipeline

pipe = StableAudioPipeline.from_pretrained("stabilityai/stable-audio-open-1.0", torch_dtype=torch.float16)
pipe = pipe.to("cuda")

# define the prompts
prompt = "The sound of a hammer hitting a wooden surface."
negative_prompt = "Low quality."

# set the seed for generator
generator = torch.Generator("cpu").manual_seed(0)

# run the generation
audio = pipe(
    prompt,
    negative_prompt=negative_prompt,
    num_inference_steps=200,
    audio_end_in_s=10.0,
    num_waveforms_per_prompt=3,
    generator=generator,
).audios

output = audio[0].T.float().cpu().numpy()
sf.write("hammer.wav", output, pipe.vae.sampling_rate)


In [ ]:
# !pip install diffusers[torch] accelerate scipy
from diffusers import DiffusionPipeline
from scipy.io.wavfile import write

model_id = "harmonai/glitch-440k"
pipe = DiffusionPipeline.from_pretrained(model_id)
pipe = pipe.to("cpu")

audios = pipe(audio_length_in_s=4.0).audios

# To save locally
for i, audio in enumerate(audios):
    write(f"test_{i}.wav", pipe.unet.sample_rate, audio.transpose())
    
# To dislay in google colab
import IPython.display as ipd
for audio in audios:
    display(ipd.Audio(audio, rate=pipe.unet.sample_rate))


In [ ]:
from transformers import pipeline

pipe = pipeline("text-to-speech", model="suno/bark-small")

In [ ]:
text = "my name is fawaz and iam a bhadwa huuuuuuuu "
output = pipe(text)

In [ ]:
from IPython.display import Audio

Audio(output["audio"], rate=output["sampling_rate"])

## AUDIOGEN

In [ ]:
# !pip install git+https://github.com/facebookresearch/audiocraft.git
# !pip install audiocraft

In [ ]:
import torchaudio
from audiocraft.models import AudioGen
from audiocraft.data.audio import audio_write

model = AudioGen.get_pretrained('facebook/audiogen-medium')
model.set_generation_params(duration=5)  # generate 5 seconds.
descriptions = ['hi my name is sarvesh.']
wav = model.generate(descriptions)  # generates 3 samples.

for idx, one_wav in enumerate(wav):
    # Will save under {idx}.wav, with loudness normalization at -14 db LUFS.
    audio_write(f'{idx}', one_wav.cpu(), model.sample_rate, strategy="loudness", loudness_compressor=True)


In [ ]:
#https://imagen.research.google/

#https://prompthero.com/interior-design-prompts